In [ ]:
# Optional examples below assume runner/context already contain aligned sample state.
import os

import sign_alignment.pipeline as pp
from sign_alignment.data_source import PrototypeSource
from sign_alignment.dift_align import DiftAlignmentConfig, DiftRuntime


In [ ]:
# Supplement 1: keep detector geometry and only relabel matched detections.
runner.run([
    pp.Step(
        "Result without optimization",
        pp.create_result_without_optimization,
        pp.vis_result_without_optimization,
    ),
    pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info),
    pp.Step("Offset analysis", lambda _: None, pp.vis_offset_analysis),
])


In [ ]:
# Supplement 2: run PSR directly from the coarse aligned boxes.
runner.run([
    pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer),
    pp.Step("Optimize PSR", pp.optimize_psr, pp.vis_optimization),
    pp.Step("Loss history", lambda _: None, pp.vis_loss_history),
    pp.Step("Results comparison", lambda _: None, pp.vis_results_comparison),
    pp.Step("Parameter changes", lambda _: None, pp.vis_parameter_changes),
])


In [ ]:
# Supplement 3: configure prototype images and DIFT on the existing context.
DIFT_CHECKPOINT = os.path.expanduser("~/erc-src/ProtoSnap/weights/SD_with_prompt")
CANONICAL_FEATURE_DIR = os.path.expanduser(
    "~/erc-work-data/signs_alignment_data/precompute_feautures"
)
context.dift = DiftRuntime(
    checkpoint=DIFT_CHECKPOINT,
    feature_dir=CANONICAL_FEATURE_DIR,
    config=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
)
context.sign_source = PrototypeSource()
runner.run([
    pp.Step("Setup source signs", pp.setup_source_signs, pp.vis_source_signs),
])


In [ ]:
# Replace align_text_rows with this DIFT sliding-window step when testing feature search.
feature_config = pp.FeatureCoarseAlignmentConfig(
    step_px=100,
    search_margin_px=100,
    assignment_min_score=0.0,
)
runner.run([
    pp.Step(
        "DIFT sliding-window coarse alignment",
        lambda ctx: pp.align_text_rows_with_feature_search(ctx, feature_config),
        pp.vis_feature_coarse_alignment,
    ),
])
feature_run = pp.get_feature_coarse_run(context)


In [ ]:
# Combine PSR with prototype overlays and the optional DIFT affine probe.
runner.run([
    pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer),
    pp.Step("Optimize until DIFT probe", pp.optimize_psr_until_dift_probe, pp.vis_optimization),
    pp.Step("Source sign overlay", pp.create_source_sign_overlay, pp.vis_source_sign_overlay),
    pp.Step("DIFT affine probe", pp.run_dift_affine_probe, pp.vis_dift_affine_probe),
    pp.Step("Finish PSR optimization", pp.optimize_psr_after_dift_probe, pp.vis_optimization),
])
